# 检测常用网络模块 Demo

本 Notebook 演示 **Backbone**、**Neck**、**感受野模块**、**注意力模块** 的简化实现与用法。

**整体流程：** 输入图像 → Backbone →（可选 SPP/ASPP）→ Neck(FPN/PAN) → Head。Backbone/Neck 中可插入 SE/SAM 等注意力。

In [ ]:
import sys
import os
# 将 det 目录加入 path（若当前目录为 Module，则其父目录即为 det）
sys.path.insert(0, os.path.dirname(os.path.abspath('.')))

import torch
from Module.backbone import (
    build_resnet, build_darknet, build_cspdarknet, build_efficientrep,
)
from Module.neck import build_fpn, build_pan, build_reppan, build_mpan
from Module.receptive_field import SPP, ASPP, RFB
from Module.attention import SEBlock, SAM, CBAM
from Module.blocks import ELAN1, EELAN, RELAN, RepNCSPELAN4, C3k2, C2PSA, RepConv

## 1. Backbone

- **ResNet**：残差连接，输出 C2~C5 多阶段特征。
- **DarkNet**：YOLOv2/v3 风格，3×3/1×1 + 残差块。
- **CSPDarkNet**：CSP 结构，减少计算、梯度更均衡。

In [ ]:
x = torch.randn(2, 3, 224, 224)

resnet = build_resnet(50)
feats_r = resnet(x)
print('ResNet-50 多阶段输出:')
for i, f in enumerate(feats_r):
    print(f'  C{i+2}: {f.shape}')

darknet = build_darknet()
feats_d = darknet(torch.randn(2, 3, 416, 416))
print('\nDarkNet 多阶段输出:')
for i, f in enumerate(feats_d):
    print(f'  Stage{i+2}: {f.shape}')

csp = build_cspdarknet()
feats_c = csp(torch.randn(2, 3, 416, 416))
print('\nCSPDarkNet 多阶段输出:')
for i, f in enumerate(feats_c):
    print(f'  Stage{i+1}: {f.shape}')

## 2. Neck：FPN / PAN

将 Backbone 的多阶段特征融合为统一通道的多尺度特征，供检测头使用。

In [ ]:
in_chs = [256, 512, 1024, 2048]
fpn = build_fpn(in_chs, out_channels=256)
pan = build_pan(in_chs, out_channels=256)

feats = [torch.randn(2, c, 56//(2**i), 56//(2**i)) for i, c in enumerate(in_chs)]
fpn_outs = fpn(feats)
pan_outs = pan(feats)
print('FPN 输出:', [o.shape for o in fpn_outs])
print('PAN 输出:', [o.shape for o in pan_outs])

## 3. 感受野增强：SPP / ASPP / RFB

在单层特征上做多尺度池化或空洞卷积，扩大感受野。

In [ ]:
c, h, w = 256, 14, 14
t = torch.randn(2, c, h, w)

spp = SPP(c, 256)
aspp = ASPP(c, 256)
rfb = RFB(c, 256)
print('SPP  out:', spp(t).shape)
print('ASPP out:', aspp(t).shape)
print('RFB  out:', rfb(t).shape)

## 4. 注意力：SE / SAM / CBAM

SE 做通道重标定，SAM 做空间重标定，CBAM 为两者串联。

In [ ]:
x = torch.randn(2, 64, 28, 28)
print('SE   out:', SEBlock(64)(x).shape)
print('SAM  out:', SAM()(x).shape)
print('CBAM out:', CBAM(64)(x).shape)

## 5. 新增模块（按时间线）

- **EfficientRep（YOLOv6, 2022）**：RepConv/RepBlock 作为 backbone 计算块。
- **Rep-PAN（YOLOv6, 2022）**：在 PAN 拓扑上用 RepBlock 融合，推理可融合为单路径卷积。
- **MPAN（工程变体）**：把 PAN 做多次路径聚合（multi-pass）。
- **ELAN / GELAN（YOLOv7/YOLOv9）**：层聚合 block 家族（这里演示 `ELAN1`、`RepNCSPELAN4`）。
- **C3k2 / C2PSA（工程常见）**：Ultralytics 风格 block 的简化实现。

In [ ]:
# EfficientRep backbone + Rep-PAN neck
backbone = build_efficientrep(
    in_channels=3,
    channels_list=(64, 128, 256, 512, 1024),
    num_repeats=(1, 1, 3, 3, 1),
    use_spp=True,
)

x = torch.randn(1, 3, 640, 640)
c3, c4, c5 = backbone(x)
print('EfficientRep outputs:', c3.shape, c4.shape, c5.shape)

neck = build_reppan(in_channels=(c3.shape[1], c4.shape[1], c5.shape[1]), out_channels=256)
p3, p4, p5 = neck((c3, c4, c5))
print('Rep-PAN outputs:', p3.shape, p4.shape, p5.shape)

# MPAN（多次 PAN）示例：这里直接对 (p3,p4,p5) 走两次 PAN 形态
mpan = build_mpan([p3.shape[1], p4.shape[1], p5.shape[1]], out_channels=256, num_passes=2)
mp_feats = mpan([p3, p4, p5])
print('MPAN outputs:', [t.shape for t in mp_feats])

# ELAN / E-ELAN / GELAN / R-ELAN blocks
elan = ELAN1(256, 256, c3=256, c4=128)
eelan = EELAN(256, 256, c3=256, c4=128, groups=2)
relan = RELAN(256, 256, c3=256, c4=128)
gel = RepNCSPELAN4(256, 256, c3=256, c4=128, n=1)
print('ELAN1:', elan(torch.randn(1, 256, 80, 80)).shape)
print('EELAN:', eelan(torch.randn(1, 256, 80, 80)).shape)
print('RELAN:', relan(torch.randn(1, 256, 80, 80)).shape)
print('RepNCSPELAN4:', gel(torch.randn(1, 256, 80, 80)).shape)

# C3k2 / C2PSA blocks
c3k2 = C3k2(256, 256, n=2, c3k=True)
c2psa = C2PSA(256, 256, n=2)
print('C3k2:', c3k2(torch.randn(1, 256, 80, 80)).shape)
print('C2PSA:', c2psa(torch.randn(1, 256, 80, 80)).shape)

# RepConv（示意：可切换 deploy）
rc = RepConv(64, 64, deploy=False)
y = rc(torch.randn(1, 64, 80, 80))
rc.switch_to_deploy()
y2 = rc(torch.randn(1, 64, 80, 80))
print('RepConv train/deploy OK:', y.shape, y2.shape)